<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# Notebook 03: Foundation Model Training

We train a **decoder foundation model** on the tokenized transaction sequences from notebook 02 using **causal language modeling** (next-token prediction). At each position, the model predicts the next token from all preceding tokens, learning transaction patterns without any fraud labels.

```
Input:  <bos>  AMT_3  MERCH_1498  CAT_RETAIL  MCC_5912  HOUR_05  DOW_0  ...
Target:        AMT_3  MERCH_1498  CAT_RETAIL  MCC_5912  HOUR_05  DOW_0  MONTH_05  ...
```

Every token position provides a training signal, so through millions of predictions the model builds rich representations of spending patterns, merchant relationships, and temporal behaviors.

## Model Configuration

| Parameter | Value | Notes |
|-----------|-------|-------|
| **Architecture** | Decoder-only Transformer | Causal (autoregressive) attention |
| **Hidden Size** | 512 | Transformer hidden dimension |
| **Layers** | 8 | Transformer depth |
| **Attention Heads** | 8 query / 2 KV | Grouped-Query Attention (GQA) |
| **FFN Size** | 1408 | SwiGLU activation |
| **Context Window** | 8192 | RoPE positional encoding |
| **Parameters** | **\~29M** | Compact embedding table |
| **Vocab Size** | \~6,252 | Matches `FinancialTokenizerPipeline` (merchant_hash_size=2,000) |
| **Training Data** | \~263M tokens | \~64K sequences from 19.5M transactions |

## Training Pipeline

Training uses `scripts/train_decoder_model.py` with **NeMo AutoModel** and FSDP2 for distributed training. Configuration is defined in `configs/pretrain_financial_decoder.yaml`; custom tokenizer and dataset are integrated via the `_target_` syntax. Scaling from 1 GPU to multi-node requires only a `torchrun` wrapper -- no code changes.

## Custom Components

We only provide two domain-specific components -- everything else (FSDP2, optimizer, checkpointing, mixed precision) is handled by NeMo AutoModel:

1. **`FinancialTabularTokenizer`** -- custom tokenization for transaction data
2. **`FinancialCLMDataset`** -- next-token prediction data loading (referenced in YAML via `_target_`)

Checkpoints are saved in HuggingFace format (`save_consolidated: true`), enabling downstream use with `AutoModelForCausalLM.from_pretrained()`.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch.cuda")
warnings.filterwarnings("ignore", category=SyntaxWarning, module="liger_kernel")
warnings.filterwarnings("ignore", message="The '.*' attribute with value.*was provided")

import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data_ieee"
CORPUS_DIR = DATA_DIR / "decoder_corpus"
CORPUS_PATH = CORPUS_DIR / "train_corpus.txt"
VAL_CORPUS_PATH = CORPUS_DIR / "val_corpus.txt"

YAML_CONFIG = PROJECT_ROOT / "configs/pretrain_financial_decoder.yaml"
TRAIN_SCRIPT = PROJECT_ROOT / "scripts/train_decoder_model.py"

PRETRAINED_MODEL = PROJECT_ROOT / "models_ieee/decoder-foundation-model"

# Verify prerequisites from notebook 02
assert CORPUS_PATH.exists(), f"Training corpus not found: {CORPUS_PATH}  (run notebook 02 first)"
assert VAL_CORPUS_PATH.exists(), f"Validation corpus not found: {VAL_CORPUS_PATH}  (run notebook 02 first)"
assert YAML_CONFIG.exists(), f"YAML config not found: {YAML_CONFIG}"
assert TRAIN_SCRIPT.exists(), f"Training script not found: {TRAIN_SCRIPT}"

# Pre-trained checkpoint (used by notebook 04 for inference)
if PRETRAINED_MODEL.exists() and (PRETRAINED_MODEL / "config.json").exists():
    print(f"Pre-trained decoder checkpoint available for notebook 04:")
    print(f"  {PRETRAINED_MODEL}\n")

print(f"Training corpus : {CORPUS_PATH}")
print(f"Validation      : {VAL_CORPUS_PATH} ({'exists' if VAL_CORPUS_PATH.exists() else 'NOT FOUND'})")
print(f"YAML config     : {YAML_CONFIG}")
print(f"Pre-trained     : {PRETRAINED_MODEL}")

Training corpus : /export/home/sobhan/cs25mtech14005/transaction-foundation-model/data_ieee/decoder_corpus/train_corpus.txt
Validation      : /export/home/sobhan/cs25mtech14005/transaction-foundation-model/data_ieee/decoder_corpus/val_corpus.txt (exists)
YAML config     : /export/home/sobhan/cs25mtech14005/transaction-foundation-model/configs/pretrain_financial_decoder.yaml
Pre-trained     : /export/home/sobhan/cs25mtech14005/transaction-foundation-model/models_ieee/decoder-foundation-model


## 1. Launch Training

Training is configured through `configs/pretrain_financial_decoder.yaml` (model architecture, AdamW + cosine annealing optimizer, FSDP2 distribution, consolidated HF checkpoints). The cell below runs a **30-step demo** on 1 GPU (\~2 min). The pre-trained model used in notebook 04 was trained for \~3,000 steps on 8x A100.

The random baseline loss is `ln(~6,252) ~ 8.7`; loss should drop to \~5–6 within 30 steps.

### YAML Configuration

In [2]:
print(f"=== {YAML_CONFIG} ===\n")
with open(YAML_CONFIG) as f:
    print(f.read())

=== /export/home/sobhan/cs25mtech14005/transaction-foundation-model/configs/pretrain_financial_decoder.yaml ===

# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Financial Foundation Model Pretraining — Decoder-Only
#
# NeMo AutoModel configuration for pretraining a decoder foundation model on
# financial transaction sequences using causal language modeling.
# Swap

In [3]:
import subprocess
import sys
import torch
import os
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6,7"

DEMO_CHECKPOINT_DIR = PROJECT_ROOT / "models" / "ieee-native-decoder" / "checkpoints"
DEMO_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

n_gpus = torch.cuda.device_count()
local_batch_size = 16
global_batch_size = local_batch_size * n_gpus
max_steps = 472006 // global_batch_size
val_every = max_steps // 10

cmd = [
    "torchrun", f"--nproc-per-node={n_gpus}", str(TRAIN_SCRIPT),
    "-c", str(YAML_CONFIG),
    "--dataset.data_path", str(CORPUS_PATH),
    "--validation_dataset.data_path", str(VAL_CORPUS_PATH),
    "--step_scheduler.max_steps", str(max_steps),
    "--step_scheduler.global_batch_size", str(global_batch_size),
    "--step_scheduler.local_batch_size", str(local_batch_size),
    "--lr_scheduler.lr_warmup_steps", str(max_steps // 10),
    "--step_scheduler.val_every_steps", str(val_every),
    "--step_scheduler.ckpt_every_steps", str(max_steps // 3),
    "--checkpoint.checkpoint_dir", str(DEMO_CHECKPOINT_DIR),
]

print(f"Launching training ({max_steps} steps, {n_gpus} GPUs):")
print(" ".join(cmd))
print()

result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    print(f"\nTraining exited with code {result.returncode}")


Launching training (7375 steps, 4 GPUs):
torchrun --nproc-per-node=4 /export/home/sobhan/cs25mtech14005/transaction-foundation-model/scripts/train_decoder_model.py -c /export/home/sobhan/cs25mtech14005/transaction-foundation-model/configs/pretrain_financial_decoder.yaml --dataset.data_path /export/home/sobhan/cs25mtech14005/transaction-foundation-model/data_ieee/decoder_corpus/train_corpus.txt --validation_dataset.data_path /export/home/sobhan/cs25mtech14005/transaction-foundation-model/data_ieee/decoder_corpus/val_corpus.txt --step_scheduler.max_steps 7375 --step_scheduler.global_batch_size 64 --step_scheduler.local_batch_size 16 --lr_scheduler.lr_warmup_steps 737 --step_scheduler.val_every_steps 737 --step_scheduler.ckpt_every_steps 2458 --checkpoint.checkpoint_dir /export/home/sobhan/cs25mtech14005/transaction-foundation-model/models/ieee-native-decoder/checkpoints



W0718 14:56:45.105000 4054723 site-packages/torch/distributed/run.py:766] 
W0718 14:56:45.105000 4054723 site-packages/torch/distributed/run.py:766] *****************************************
W0718 14:56:45.105000 4054723 site-packages/torch/distributed/run.py:766] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0718 14:56:45.105000 4054723 site-packages/torch/distributed/run.py:766] *****************************************
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.7.1+cu118).
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.7.1+cu118).
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.7.1+cu118).
Skipping import of cpp extensions due

grouped_gemm is not available. Please run:pip install git+https://github.com/fanshiqing/grouped_gemm@v1.1.4grouped_gemm is not available. Please run:pip install git+https://github.com/fanshiqing/grouped_gemm@v1.1.4



/export/home/sobhan/.conda/envs/tfm/lib/python3.10/site-packages/megatron_fsdp/mixed_precision.py:114: UserWarning: Transformer Engine and Apex are not installed. Falling back to local implementations of multi_tensor_applier and multi_tensor_scale
  warnings.warn(


cfg-path: /export/home/sobhan/cs25mtech14005/transaction-foundation-model/configs/pretrain_financial_decoder.yamlcfg-path: /export/home/sobhan/cs25mtech14005/transaction-foundation-model/configs/pretrain_financial_decoder.yaml


Financial Foundation Model — Decoder-Only Pretraining
  Architecture: LlamaConfig
  Hidden size:  512
  Layers:       8
  Vocab size:   6251
  Max steps:    7375
  Batch size:   64 global, 16 local


Financial Foundation Model — Decoder-Only Pretraining
  Architecture: LlamaConfig
  Hidden size:  512
  Layers:       8
  Vocab size:   6251
  Max steps:    7375
  Batch size:   64 global, 16 local

grouped_gemm is not available. Please run:pip install git+https://github.com/fanshiqing/grouped_gemm@v1.1.4
grouped_gemm is not available. Please run:pip install git+https://github.com/fanshiqing/grouped_gemm@v1.1.4
cfg-path: /export/home/sobhan/cs25mtech14005/transaction-foundation-model/configs/pretrain_financial_decoder.yaml

Financial Foundation Model — Decoder-Only

...

2026-07-18 15:35:54 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0/4: Consolidating safetensors files from /export/home/sobhan/cs25mtech14005/transaction-foundation-model/models/ieee-native-decoder/checkpoints/epoch_4_step_6614/model to /export/home/sobhan/cs25mtech14005/transaction-foundation-model/models/ieee-native-decoder/checkpoints/epoch_4_step_6614/model/consolidated
2026-07-18 15:35:54 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0: Assigned 0 output files out of 1 total files
2026-07-18 15:35:54 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0: Done consolidating. Processed 0 unique indices in 0.01 secs.
2026-07-18 15:35:54 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0: Waiting for all ranks to complete...
2026-07-18 15:35:55 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_

Saving checkpoint to /export/home/sobhan/cs25mtech14005/transaction-foundation-model/models/ieee-native-decoder/checkpoints/epoch_5_step_7373


2026-07-18 15:40:42 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0/4: Consolidating safetensors files from /export/home/sobhan/cs25mtech14005/transaction-foundation-model/models/ieee-native-decoder/checkpoints/epoch_5_step_7373/model to /export/home/sobhan/cs25mtech14005/transaction-foundation-model/models/ieee-native-decoder/checkpoints/epoch_5_step_7373/model/consolidated
2026-07-18 15:40:42 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0: Assigned 0 output files out of 1 total files
2026-07-18 15:40:42 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0: Done consolidating. Processed 0 unique indices in 0.04 secs.
2026-07-18 15:40:42 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0: Waiting for all ranks to complete...
2026-07-18 15:40:42 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_

Saving checkpoint to /export/home/sobhan/cs25mtech14005/transaction-foundation-model/models/ieee-native-decoder/checkpoints/epoch_5_step_7374


2026-07-18 15:41:14 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0/4: Consolidating safetensors files from /export/home/sobhan/cs25mtech14005/transaction-foundation-model/models/ieee-native-decoder/checkpoints/epoch_5_step_7374/model to /export/home/sobhan/cs25mtech14005/transaction-foundation-model/models/ieee-native-decoder/checkpoints/epoch_5_step_7374/model/consolidated
2026-07-18 15:41:14 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0: Assigned 0 output files out of 1 total files
2026-07-18 15:41:14 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0: Done consolidating. Processed 0 unique indices in 0.01 secs.
2026-07-18 15:41:14 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_safetensors | Rank 0: Waiting for all ranks to complete...
2026-07-18 15:41:14 | INFO | nemo_automodel.components.checkpoint._backports.consolidate_hf_

## Checkpoints

With `save_consolidated: true`, checkpoints are in HuggingFace format (`config.json` + `safetensors`), loadable via `AutoModelForCausalLM.from_pretrained()`. The pre-trained checkpoint at `models_ieee/decoder-foundation-model/` is ready for inference in notebook 04.

## Summary

We trained a \~29M parameter decoder foundation model on \~64K tokenized transaction sequences using NeMo AutoModel. The model learns to predict the next transaction token from context through self-supervised learning, without any fraud labels. The full pre-trained checkpoint (\~3,000 steps on 8x A100) is available for the next step.

Continue to [04_inference_embedding_extraction.ipynb](./04_inference_embedding_extraction.ipynb).